# 社区糖尿病风险筛查

## 1. 项目概述

回顾：前面已经讲了一些规范，Pipeline、网格搜索、MLflow 实验追踪 等。

完整的项目——社区糖尿病风险筛查。手里只有一个文件 `incident.db`，要交出去的是一个能给每次随访打风险分、能按 15% 名额排出复查名单的模型。

### 背景

某市社区慢病随访平台运行了三年，沉淀了三类数据：居民档案、每次随访的问卷与体测、随访期间做的化验。卫健部门希望每次随访结束后，给这位居民打一个糖尿病风险分，按分数从高到低安排糖耐量复查。每批名额大约占当批随访人数的 15%，名额之外的人排到下一批。现在这件事靠医生经验，看得准的医生一天只能看几十个人，而且标准不一。我们要把它变成一个可以批量执行、可以被检查的模型。

### 预测目标

对一次随访，预测这位居民在这次随访之后 90 天内的化验是否达到阳性标准（空腹血糖 ≥ 7.0 mmol/L，或糖化血红蛋白 ≥ 6.5%），达到为 1，否则为 0。

### 约束条件

任务书还给了两条约束：
- 模型的输入只能来自这次随访本身记录的信息（问卷、体测、病史）；
- 上线时模型只能看到这次随访的记录，随访之后的化验、下次随访的记录，都不能作为输入。

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sqlite3                                              # 读 SQLite 数据库（Python 自带，不用装）
import json                                                 # 写 metrics.json、特征清单
import joblib                                               # 保存整条 Pipeline

from sklearn.model_selection import train_test_split        # 划分数据集用的
from sklearn.pipeline import Pipeline                       # 搭建流水线
from sklearn.compose import ColumnTransformer               # 数值列、类别列分开处理
from sklearn.impute import SimpleImputer                    # 填补缺失值
from sklearn.preprocessing import OneHotEncoder             # 对数据做独热编码
from sklearn.preprocessing import StandardScaler            # 对数据做标准化
from sklearn.linear_model import LogisticRegression         # 逻辑回归
from sklearn.ensemble import RandomForestClassifier         # 随机森林
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score          # 三个常用指标
from sklearn.metrics import confusion_matrix, classification_report          # 混淆矩阵、分类报告

pd.set_option("display.max_columns", 40)                    # 列多，全部显示
pd.set_option("display.width", 120)

In [27]:
SEED = 42 
DATA = "../data/"
DB = DATA + "incident.db"
# 设置后期使用的标签列的列名
TARGET = "label"

In [28]:
# with 语句: 自动打开数据库连接，代码块结束后自动关闭连接，不用手动写 conn.close()，防止忘记关闭
with sqlite3.connect(DB) as conn:
    # 从 sqlite 的系统表 sqlite_master 查询，拿到数据库里所有普通数据表的名字
    tables = pd.read_sql("select name from sqlite_master where type='table'", conn)
    print(tables)

    # 循环遍历每一张表的表名，查看表里面的信息
    for name in tables['name']:
        # PRAGMA table_info(表名)：SQLite 专属命令，查询这张表的字段信息（列名、类型、是否主键等）
        cols = pd.read_sql(f"PRAGMA table_info({name})",conn)
        # COUNT(*) 统计表总行数；取出结果里n这一列第0个值，就是行数
        n = pd.read_sql(f"select count(1) as n from {name}",conn)["n"][0]
        # 格式化打印输出：表名左对齐占12字符、行数右对齐、列数量，最后拼接所有列名
        print(f"{name:<12}{n:>8}行 {len(cols):>2}列: ", ", ".join(cols["name"]))

          name
0    residents
1       visits
2  lab_results
residents      36000行  7列:  resident_id, gender, birth_date, education, income_level, district, enroll_date
visits        101297行 23列:  visit_id, resident_id, visit_date, height_cm, weight_kg, waist_cm, sbp, dbp, smoking_status, drinking_freq, exercise_min_week, sleep_hours, veg_fruit_daily, family_history_dm, hypertension_dx, hyperlipidemia_dx, gdm_history, polyuria, polydipsia, weight_loss_unexplained, known_diabetes, created_at, chronic_mgmt_tag
lab_results    79358行  6列:  lab_id, resident_id, sample_date, report_date, fpg_mmol_l, hba1c_pct


In [29]:
# parse_dates=[...] 在读的时候就把日期列转成 datetime，后面做日期加减不用再 pd.to_datetime
with sqlite3.connect(DB) as conn:
    residents = pd.read_sql("select * from residents",conn,parse_dates=['birth_date','enroll_date'])
    visits = pd.read_sql("select * from visits",conn,parse_dates=['visit_date','created_at'])
    labs = pd.read_sql("select * from lab_results",conn,parse_dates=['sample_date','report_date'])
    print(visits)
    print(labs)
    print(residents.shape,visits.shape,labs.shape)

         visit_id resident_id visit_date  height_cm  weight_kg  waist_cm    sbp   dbp smoking_status drinking_freq  \
0       V00000001    R0000131 2023-01-01      164.1       61.4      78.6  100.0  65.0          never      frequent   
1       V00000002    R0000342 2023-01-01      162.2       73.3      91.9  115.0  62.0          never    occasional   
2       V00000003    R0000387 2023-01-01      161.5       56.0       NaN  118.0  72.0        current         never   
3       V00000004    R0000736 2023-01-01      172.0       70.2      80.6  132.0  84.0        current    occasional   
4       V00000005    R0001057 2023-01-01      156.0       48.5      63.1  125.0  69.0          never         never   
...           ...         ...        ...        ...        ...       ...    ...   ...            ...           ...   
101292  V00101293    R0034982 2025-06-30      161.6       47.9      76.3   75.0  58.0          never    occasional   
101293  V00101294    R0035219 2025-06-30      161.5     

In [30]:
visits.head(3).T

,0,1,2
visit_id,V00000001,V00000002,V00000003
resident_id,R0000131,R0000342,R0000387
visit_date,2023-01-01 00:00:00,2023-01-01 00:00:00,2023-01-01 00:00:00
height_cm,164.1,162.2,161.5
weight_kg,61.4,73.3,56.0
waist_cm,78.6,91.9,NaN
sbp,100.0,115.0,118.0
dbp,65.0,62.0,72.0
smoking_status,never,never,current
drinking_freq,frequent,occasional,never


In [31]:
visits.describe(include="number").T[['count', 'mean', 'min', '50%', 'max']].round(1)

,count,mean,min,50%,max
height_cm,101297.0,162.3,1.4,163.0,191.3
weight_kg,101297.0,64.7,31.1,64.0,198.4
waist_cm,86036.0,81.7,39.1,81.6,123.4
sbp,101297.0,120.5,0.0,121.0,176.0
dbp,101297.0,74.5,36.0,74.0,111.0
exercise_min_week,101297.0,132.1,0.0,82.0,10080.0
sleep_hours,93165.0,6.9,3.0,6.9,11.3
veg_fruit_daily,101297.0,0.6,0.0,1.0,1.0
hypertension_dx,101297.0,0.1,0.0,0.0,1.0
hyperlipidemia_dx,101297.0,0.1,0.0,0.0,1.0


### 问题：看 min、max
身高最小 1.4（这是按"米"录的）、身高最高 191.3（这是按"厘米"录的），体重最大 198.4、收缩压最小 0、每周运动最多 10,080 分钟。**单位不统一**。

In [32]:
# 查看 visits 表中每一列的确实比例
print(visits.isna().mean().round(3).sort_values(ascending=False).head(4))

gdm_history                0.485
waist_cm                   0.151
sleep_hours                0.080
weight_loss_unexplained    0.030
dtype: float64


### 解读
- `gdm_history` 0.485：妊娠糖尿病史，近一半数据缺失（48.5%）
- `waist_cm` 0.151：腰围，15.1% 缺失
- `sleep_hours` 0.080：睡眠时长，8% 缺失

In [33]:
# `duplicated()` 判断是不是和前面一模一样的整行
print("完全重复的行：",visits.duplicated().sum())

完全重复的行： 0


In [34]:
# 根据性别统计行数
print(residents['gender'].value_counts().to_dict())

# 吸烟状态
print(visits['smoking_status'].value_counts().to_dict())

# 饮酒频率统计
print(visits["drinking_freq"].value_counts().to_dict())

# 删除完全重复的
visits = visits.drop_duplicates()
# 统一性别的类别标签，最终留下：F / M (注：底部被截断)

{'F': 18276, 'M': 17032, 'male': 186, 'female': 174, '女': 172, '男': 160}
{'never': 66495, 'current': 22022, 'former': 8615, 'never ': 961, '从不': 956, 'Never': 930, 'Current': 473, '吸烟': 471, 'Former': 203, '已戒': 171}
{'never': 49524, 'occasional': 34237, 'frequent': 14436, '不喝': 1534, '偶尔': 1097, '经常': 469}


### 问题：同一个性别，写法五花八门
英文大写简写 F/M、英文全写 male/female、中文"男/女"混在一起，属于**类别不一致**，后面建模必须统一。

### 问题：吸烟状态同样混乱
never / Never / never （末尾带空格）/ 从不 其实都是【从不吸烟】；current / Current / 吸烟 是【正在吸烟】，标签不统一。

### 问题：饮酒频率中英文混写
never 和不喝是一类、occasional 和偶尔一类、frequent 和经常一类。

## 3. 数据清洗

回顾：刚才检查到一些脏数据，现在需要进行数据的清洗，例如：统一类别的标签。

### 3.1 统一类别标签

In [35]:
# 删除完全重复的
visits = visits.drop_duplicates()

# 统一性别的类别标签，最终留下：F / M
residents['gender'] = residents['gender'].replace(
    {"male":"M","男":"M","female":"F","女":"F"}
)

# 统一吸烟状态的类别标签，最终留下：英文 never，current，former
visits['smoking_status'] = visits['smoking_status'].replace(
    {'从不':"never", 
    '吸烟':"current", 
    '已戒':'former', 
    'Never':'never', 
    'Current':'current', 
    'Former':'former', 
    'never ':'never'}
    )

# 统一饮酒频率的类别标签，最终留下：never\occasional\frequent
# 【注意】原代码末尾缺少一个右括号 `)`
visits["drinking_freq"] = visits["drinking_freq"].replace(
    {"不喝":"never", 
    "偶尔":"occasional", 
    "经常":"frequent"}
    )

print(residents['gender'].value_counts().to_dict())
print(visits['smoking_status'].value_counts().to_dict())
print(visits["drinking_freq"].value_counts().to_dict())

{'F': 18622, 'M': 17378}
{'never': 69342, 'current': 22966, 'former': 8989}
{'never': 51058, 'occasional': 35334, 'frequent': 14905}


### 3.2 单位、哨兵值、越界

这一段是数值型字段清洗，专门处理三类脏数据：
1. **单位混录**（有的人按米填身高、有的人按斤填体重）
2. **哨兵值**（用 0 代表"没测/缺失"，不是真实测量值）
3. **越界异常值**（现实里不可能出现的离谱数字）

- `sbp` 收缩压（高压），血压不可能等于 0——这里的 0 不是测量结果，是录入人员标记：这个数据缺失、没有测量，这种用来代表缺失的特殊数字，就叫**哨兵值**。处理：把 0 改成 `np.nan`，统一为 pandas 标准缺失值，方便后续缺失值处理。
- `exercise_min_week` 每周运动分钟数：3000 分钟 = 50 小时，一周总共才 168 小时，一周运动 50 小时不符合常理，属于**越界异常值**，直接标记成缺失 NaN。

In [36]:
# 按米录入的身高 → 厘米
visits.loc[visits['height_cm']<3,'height_cm'] *=100
visits[visits['height_cm']<3]

# 按斤录入的体重 → 公斤
visits.loc[visits['weight_kg'] > 150,'weight_kg'] /= 2

# sbp = 收缩压（高压），血压不可能等于 0
visits.loc[visits['sbp'] == 0,'sbp'] = np.nan

visits.loc[visits['exercise_min_week'] > 3000,'exercise_min_week'] = np.nan

visits[[
    'height_cm', 
    'weight_kg', 
    'sbp', 
    'exercise_min_week'
    ]].describe().T[
        ['min', 
        '50%', 
        'max']]

,min,50%,max
height_cm,134.4,163.0,191.3
weight_kg,31.1,64.0,149.7
sbp,62.0,121.0,176.0
exercise_min_week,0.0,82.0,2162.0


清洗后验证：身高 min 从 1.4 → 134.4（单位统一），体重 max 从 198.4 → 149.7（按斤的已转公斤），收缩压 min 从 0 → 62.0（哨兵值已处理），运动 min 从 10080 → 2162（越界异常值已处理）。

In [37]:
# 一次随访 × 这个人的每一次化验
df = visits.merge(labs,on='resident_id',how='left')
# 条件判断：窗口以内
in_window = ((df['sample_date'] >= df['visit_date']) 
& (df['sample_date']<= df['visit_date'] + pd.Timedelta(days=90)))
# drop_duplicates('visit_id',keey='first'):同一个随访，只保留窗口内最早的那一次化验
df = df[in_window].sort_values('sample_date').drop_duplicates('visit_id',keep='first')
# 空腹血糖 fpg ≥7.0 mmol/L 或 糖化血红蛋白 hba1c ≥6.5%
# `.astype(int)` → True 变成 1，False 变成 0，做成 0/1 二分类标签，给模型用
df[TARGET] = ((df['fpg_mmol_l'] >= 7.0) | (df['hba1c_pct'] >=6.5)).astype(int)
# 拼上居民档案
df = df.merge(residents,on='resident_id',how='left')
print('有标签的访问：',len(df),"阳性率：%.3f"% df[TARGET].mean())

有标签的访问： 79705 阳性率：0.299


### 3.3 打标签：随访后 90 天内的首次化验

任务：根据一次随访，看这个人在随访之后 90 天内第一次化验，判断是否糖尿病阳性。

**总结**：101,297 条随访里，有 79,705 条在窗口内找到了化验，阳性率 29.9%。在这 79705 条能打上标签的随访样本里，大约 3 成的人后续化验达到糖尿病诊断标准。

### 3.4 派生特征：年龄、BMI

年龄 = 随访日期 - 出生日期；BMI = 体重 / (身高(米))²

In [38]:
df['age'] = (df['visit_date'] - df['birth_date']).dt.days / 365.25
df['bmi'] = df['weight_kg'] / (df['height_cm'] / 100)**2
# 统计
df[['age','bmi']].describe().T[['mean','min','max']].round(1)

,mean,min,max
age,60.1,22.0,90.5
bmi,24.2,15.0,67.9


**总结**：年龄 22 到 90 岁，平均 60 岁，是社区慢病随访的典型人群；BMI 平均 24.2。

---

## 4. 选列和切分

筛选哪些列可以放进模型做特征，是建模前的一步：区分【要丢掉的列】、【数值特征】、【类别特征】。

In [39]:
# 要丢掉的列
DROP_COLS = ['visit_id','resident_id','lab_id','visit_date','created_at','birth_date',
             'enroll_date','fpg_mmol_l','hba1c_pct','sample_date','report_date',TARGET]
# 数值特征
NUM_COLS = df.select_dtypes(include="number").columns.difference(DROP_COLS).tolist()
# 类别特征
CAT_COLS = df.select_dtypes(exclude=['number','datetime']).columns.difference(DROP_COLS).tolist()
# 所有的特征
FEATURES = NUM_COLS + CAT_COLS
print('数值列',len(NUM_COLS),NUM_COLS)
print('类别列',len(CAT_COLS),CAT_COLS)

数值列 17 ['age', 'bmi', 'dbp', 'exercise_min_week', 'gdm_history', 'height_cm', 'hyperlipidemia_dx', 'hypertension_dx', 'known_diabetes', 'polydipsia', 'polyuria', 'sbp', 'sleep_hours', 'veg_fruit_daily', 'waist_cm', 'weight_kg', 'weight_loss_unexplained']
类别列 8 ['chronic_mgmt_tag', 'district', 'drinking_freq', 'education', 'family_history_dm', 'gender', 'income_level', 'smoking_status']


### 4.1 DROP_COLS = 必须扔掉、不能进模型的列

分 4 类理解：
1. **主键 ID**：`visit_id`、`resident_id`、`lab_id`，只是编号，没有预测意义，不能喂给模型。
2. **原始日期**：`visit_date`、`birth_date` 等，我们已经用它们算出了 `age` 年龄，原始日期不再使用。
3. **泄漏字段（最重要）**：`fpg_mmol_l`（空腹血糖）、`hba1c_pct`（糖化血红蛋白）——我们的标签 `TARGET` 就是靠这两个化验值算出来的！如果把这两列放进模型，相当于直接把答案给模型，属于**数据泄露**，模型会假的特别准，上线完全没用。
4. **TARGET**：标签本身，是我们要预测的目标，当然不能当做特征输入。

### 4.2 数值特征

`select_dtypes(include="number")`：自动选出所有数字类型列，`.difference(DROP_COLS)`：去掉上面 DROP 列表里要丢弃的列。

### 4.3 类别特征

`exclude=["number", "datetime"]`：排除数字、日期，剩下的就是文本类别列。

### 4.4 划分训练集、验证集，落盘

In [40]:
# 划分
train,valid = train_test_split(df,test_size=0.3,random_state=SEED,stratify=df[TARGET])
# 把划分好的训练集、验证集保存成 parquet 文件，
# 后面训练模型的时候直接读取，不用反复跑前面清洗、特征工程代码
# index=False: 不保存行索引
train.to_parquet('train.parquet',index=False)
valid.to_parquet('valid.parquet',index=False)
# 分开特征 X 和 标签 y
X_train,y_train = train[FEATURES],train[TARGET]
X_valid,y_valid = valid[FEATURES],valid[TARGET]
print(train.shape,valid.shape)
print("阳性率 训练 %.3f 验证 %.3f" % (y_train.mean(),y_valid.mean()))

(55793, 37) (23912, 37)
阳性率 训练 0.299 验证 0.299


## 5. 建模、评估与交付

主要：sklearn Pipeline 流水线，把【缺失值填充、标准化、独热编码、逻辑回归】打包成一整条链路，是建模的标准写法。

### 5.1 流水线：预处理 + 逻辑回归

In [41]:
# 数值列：中位数填补 + 标准化；类别列：众数填补 + 独热。每个模型各造一份，不共用
def make_pre(num_col,cat_col):
    return ColumnTransformer([
        ('num',Pipeline([
            # 缺失值用中位数填充
            ('impute',SimpleImputer(strategy='median')),
            # 标准化
            ('scale',StandardScaler())
        ]),num_col),
        ('cat',Pipeline([
            # 缺失值用众数填充
            ('impute',SimpleImputer(strategy='most_frequent')),
            # 独热编码
            ('onehot',OneHotEncoder(handle_unknown='ignore'))
        ]),cat_col)
    ])
# 整条流水线
pipe = Pipeline([
    # 预处理
    ('pre',make_pre(NUM_COLS,CAT_COLS)),
    # 逻辑回归模型
    ('lr',LogisticRegression(max_iter=2000))
])
# 训练
pipe.fit(X_train,y_train)
print('预处理之后的列数：',pipe.named_steps['pre'].transform(X_train.head(1)).shape[1])

预处理之后的列数： 47


**原始特征**：17 数值 + 8 类别 = 25 列。17 个数值列：做完中位数填充 + 标准化，数量不变，还是 17 列；8 个类别列：OneHot 独热展开，变成 47 - 17 = 30 列。

### 5.2 评估逻辑回归

In [42]:
# predict_proba 输出两列：[阴性概率, 阳性概率]，[:,1] 取出【是糖尿病阳性的概率】，取值 0~1
pro = pipe.predict_proba(X_valid)[:,1]
# 阈值 0.5：概率 ≥0.5 → 预测为阳性 1；小于 0.5 → 阴性 0
pred = (pro>=0.5).astype(int)
print({
    # 准确率
    "accuracy":round(accuracy_score(y_valid,pred),4),
    # F1 分数
    "f1": round(f1_score(y_valid,pred),4),
    # AUC 值
    "roc_auc":round(roc_auc_score(y_valid,pred),4)
})
# 混淆矩阵
print(confusion_matrix(y_valid,pred))

{'accuracy': 0.9558, 'f1': 0.9244, 'roc_auc': 0.9406}
[[16400   361]
 [  695  6456]]


**指标解读**：
- `accuracy = 0.9558`（准确率 95.58%）：所有样本里预测对的比例。这个数据集阳性率只有 29.9%，单纯看准确率容易骗人，只做参考。
- `f1 = 0.9244`（F1=92.44%）：精确率和召回率的调和平均，兼顾"不误报"和"不漏检"，是不平衡分类很重要的指标，这个数值很不错。
- `roc_auc = 0.9406`（AUC=0.9406）：AUC 衡量模型区分正负样本的能力，范围 0~1，越接近 1 越好。0.9406 说明这个逻辑回归模型区分糖尿病人和非糖尿病人的能力很强，效果优秀。

**混淆矩阵格式**：`[[TN, FP], [FN, TP]]`
- TN=16400：真实阴性，模型预测阴性（正确识别健康人）
- FP=361：真实阴性，模型预测阳性（误报：健康人被错判成高危）
- FN=695：真实阳性，模型预测阴性（漏检：糖尿病人没被找出来）
- TP=6456：真实阳性，模型预测阳性（正确抓到糖尿病人）

### 5.3 流水线：预处理 + 随机森林

In [ ]:
# 整条流水线
pipe_rf = Pipeline([
    # 预处理
    ('pre',make_pre(NUM_COLS,CAT_COLS)),
    # 随机森林模型
    ('rf',RandomForestClassifier(n_estimators=200,random_state=SEED,n_jobs=1))
])
# 训练
pipe_rf.fit(X_train,y_train)
# predict_proba 输出两列：[阴性概率, 阳性概率]，[:,1] 取出【是糖尿病阳性的概率】，取值 0~1
pro_rf = pipe_rf.predict_proba(X_valid)[:,1]
# 阈值 0.5：概率 ≥0.5 → 预测为阳性 1；小于 0.5 → 阴性 0
pred_rf = (pro_rf>=0.5).astype(int)
print({
    # 准确率
    "accuracy":round(accuracy_score(y_valid,pred_rf),4),
    # F1 分数
    "f1": round(f1_score(y_valid,pred_rf),4),
    # AUC 值
    "roc_auc":round(roc_auc_score(y_valid,pred_rf),4)
})
# 打印报告
print(classification_report(y_valid,pred_rf,digits=3))

**总结**：逻辑回归 accuracy=0.9558，f1=0.9244，auc=0.9406；随机森林 accuracy=0.9596，f1=0.9312，auc=0.9468。**结论**：随机森林整体效果略好于逻辑回归。

### 5.4 两个模型放一起，挑 AUC 高的

In [ ]:
results = {
    "lr":{
        # 准确率
        "accuracy":round(accuracy_score(y_valid,pred),4),
        # F1 分数
        "f1": round(f1_score(y_valid,pred),4),
        # AUC 值
        "roc_auc":round(roc_auc_score(y_valid,pred),4) 
    },
    "rf":{
        # 准确率
        "accuracy":round(accuracy_score(y_valid,pred_rf),4),
        # F1 分数
        "f1": round(f1_score(y_valid,pred_rf),4),
        # AUC 值
        "roc_auc":round(roc_auc_score(y_valid,pred_rf),4)  
    }
}
best = max(results,key=lambda k:results[k]['roc_auc'])
best_pipe = {'lr':pipe,'rf':pipe_rf}[best]
pd.DataFrame(results).T

,accuracy,f1,roc_auc
lr,0.9558,0.9244,0.9406
rf,0.9596,0.9312,0.9468


In [ ]:
# 复盘 1：筛选的对象是谁？
# 任务书说"给居民打风险分，安排糖尿病复查"。已经确诊糖尿病的人，还需要被筛查吗？
# 考虑问题：我们筛查的目标人群到底是谁？已确诊糖尿病的人，该不该放进模型数据集？
# 目前主要关注的字段：known_diabetes：0=未确诊的居民  1=已确诊的居民

# 统计 known_diabetes 字段各个取值的数量：0=之前没有确诊糖尿病，1=已经确诊糖尿病
print(visits['known_diabetes'].value_counts().to_dict())

# 按照【是否已确诊糖尿病】分组，统计每组样本数量size、本组内标签TARGET的均值（也就是阳性率），保留3位小数
print(df.groupby('known_diabetes')[TARGET].agg(['size','mean']).round(3))

# 筛选出【未确诊人群 known_diabetes == 0】，计算这部分人群的TARGET均值（阳性率）并打印
print("剔除已确诊之后，阳性率 %.3f" % df.loc[df['known_diabetes']==0,TARGET].mean())

### 复盘1 总结：
1. 看原始数据里，有多少人是之前没有确诊、多少人已经确诊糖尿病。输出 `{0: 84029, 1: 17268}`，说明数据里混了大量已经确诊的患者。
2. 分组统计：
   - `known_diabetes=0`（未确诊居民）：样本 64577，阳性率只有 **0.143**；
   - `known_diabetes=1`（已经确诊的人）：样本 15128，阳性率高达 **0.965**；
   - 已经确诊的这批人，几乎标签都是阳性，是天然的"高分样本"。如果不剔除，模型很容易靠这个特征刷出很高 AUC，造成虚高。
3. 单独看真正需要筛查的人群（未确诊）的阳性率：**0.143**。这个 0.143 才是我们这个糖尿病筛查任务真实的基线分布，后面建模、评估指标，都应该基于这个人群，而不是带上已确诊病人。

**结论**：任务目标是给居民打分、安排糖耐复查，已经确诊糖尿病的人不属于本次筛查对象，必须从训练、测试集剔除；如果不剔除，模型学到的不是"谁会新发糖尿病"，而是识别"这个人本来就有病"，指标再好也不符合业务需求。

In [ ]:
# 复盘 2：chronic_mgmt_tag 在随访那天存在吗？
# 问题：chronic_mgmt_tag（慢病纳管标签）看起来和别的列一样，取值只有四种。它能不能当特征？
# 校验这个字段的时间因果问题（特征泄露），是机器学习里非常经典的因果校验：
# 特征必须是【随访当时就已经存在、在结局发生之前就有的信息】，不能是结局发生之后才产生的标签。

# 筛选出还没有确诊糖尿病的人群（本次筛查目标人群）
screen = df[df['known_diabetes']==0]
# 按慢病纳管标签chronic_mgmt_tag分组，统计每组样本数量size、标签TARGET均值（阳性率），保留3位小数
print(screen.groupby('chronic_mgmt_tag')[TARGET].agg(['size','mean']).round(3))

# 按居民id分组，统计每个居民拥有的不同慢病标签的数量
n_tag = visits.groupby('resident_id')['chronic_mgmt_tag'].nunique()
# 统计【每个居民标签种类数】的分布，转为字典打印
print("每个居民有几种tag:",n_tag.value_counts().to_dict())

# 提取居民R0000014的多条随访记录：随访日期、是否既往确诊糖尿病、慢病纳管标签，用来时序观察
visits.loc[visits['resident_id']=='R0000014',
['visit_date','known_diabetes','chronic_mgmt_tag']]

### 复盘2 总结：
1. 在未确诊人群里，不同慢病标签的阳性率差异巨大：
   - `both`（同时糖尿病+高血压纳管）**0.940**
   - `diabetes`（糖尿病纳管）**0.929**
   - `hypertension`（高血压纳管）**0.333**
   - `none`（无慢病纳管）**0.062**
2. 输出 `{1:36000}`：每个居民全程只有 **1 种固定 tag**，标签不会随随访时间变化。
3. 拿单个居民 `R0000014` 的时序记录是关键证据：
   - 2023-01-22：`known_diabetes=0`（当时还没确诊），但是标签已经是 `diabetes`
   - 后面几次随访：`known_diabetes=1`（已经确诊糖尿病），标签依旧是 `diabetes`
   - `chronic_mgmt_tag` 这个糖尿病纳管标签，是在这个人后续确诊糖尿病之后，才给打上的慢病管理标签，是**结果**，不是事前的基线特征。
   - 虽然在第一条随访记录里它也显示为 `diabetes`，但这个标签本质是慢病系统的纳管标识，是结局相关的后置标签，不是随访当天已经存在的前置风险因素。

**结论**：如果直接拿 `chronic_mgmt_tag` 当特征，属于**特征泄露（data leakage）**：模型不是学习风险因素，而是直接拿到了事后的慢病登记结果，指标虚高，上线之后无法使用。`chronic_mgmt_tag` 不能直接拿来做本次筛查模型的特征——这个标签是慢病管理的登记标签，和本次是否新发糖尿病这个结局强耦合，存在时间上的因果倒置，属于泄露特征。

In [ ]:
# 复盘 3：考卷是哪一年的——时间切分校验
# 问题：上线以后，模型每天面对的是"明天来随访的人"。我们的验证集，能代表明天的人吗？
# 模型上线后，预测的是"明天新来随访的居民"，随机切分得到的验证集，能不能真实模拟上线后的场景？
# 这一段是校验时间泄露、样本切分方式是否合理，是时序类风险模型非常关键的校验。

# 交叉表：统计每一年的随访数据，有多少划分到train训练集、多少划分到valid验证集
print(pd.crosstab(df['visit_date'].dt.year,np.where(df.index.isin(train.index),'train','valid')))

# 将训练集 和 验证集的居民ID、随访日期合并，同一个居民的训练随访、验证随访放到同一行
later = train.merge(valid[['resident_id','visit_date']],on='resident_id',suffixes=("","_v"))
# 计算：验证集中的居民，曾经在训练集里出现过的占比
print("验证集里的居民，在训练集里出现过的比例 %.3f" % valid["resident_id"].isin(train["resident_id"]).mean())
# 统计：同一个居民，训练集里的随访日期 > 验证集随访日期 的记录条数（训练数据用了验证时间之后的数据）
print("训练集里比同一居民的验证随访更晚的随访 %d 条" % (later["visit_date"] > later["visit_date_v"]).sum())

### 复盘3 总结：
1. 可以看到 2023、2024、2025 每一年的数据，都同时分到了训练集和验证集——普通随机切分，是打乱时间，按样本随机抽签划分，不是按时间先后切分。
2. 第二行合并居民随访记录：把同一个居民的训练随访、验证随访配对，方便对比两者的时间先后。
3. 第三行输出：验证集居民在训练集出现比例 **0.761**——意思：76.1% 的验证居民，之前已经在训练集里见过这个居民了。模型不是"第一次见这个居民"，它已经学习过这个人的历史信息，和上线场景矛盾：上线时来的人，模型应该是第一次对他做风险打分。
4. 第四行输出：训练集里比同一居民验证随访更晚的随访有 **15227 条**——这是最严重的时间泄露：验证随访是更早的时间点，训练集却拿了这个居民之后更晚的随访数据去训练。相当于考试的时候，考卷是过去的题，但是复习资料里包含了这道题之后的答案，模型作弊。

**结论**：随机切分的验证集不能代表上线后"明天来随访的人"。上线场景：时间往前走，只能用【过去的历史数据】训练，去预测【未来新的随访】；随机切分的问题：同一个人，未来的随访数据跑到训练集，验证集里很多居民已经在训练集见过。

**正确做法**：时间切分（时间回溯验证），用早于某个时间点的数据训练，用该时间点之后的数据做验证，模拟真实上线。随机切分让模型用未来训练、在过去考试。真实上线时，模型只见过过去，要预测的是未来——随机切分的验证分数回答不了"上线以后会怎样"。同一个居民出现在训练和验证两边是允许的（同一社区的人会反复来随访），禁止的是同一次随访跨集。

In [ ]:
# 5.5 出名单：15%名额
# 任务书要的不是 0/1，是一张名单：每批随访按风险分排序，前 15% 去复查。名单里有多少是真阳性？

# 给验证集valid新增一列score，score是最优模型预测的糖尿病阳性风险概率（0~1）
valid = valid.assign(score = best_pipe.predict_proba(X_valid)[:,1])
# 计算名额：总验证人数 × 15%，转为整数，也就是本次筛选最多选出15%的人复查
k = int(round(len(valid) * 0.15))
# 按score风险概率从大到小，取出风险最高的k个人，作为优先复查名单
top = valid.nlargest(k,"score")
# 打印：名额总人数、这份名单里真正是糖尿病阳性的人数
# 主要通过 label标签进行判别，是不是真的阳性
print("名额",k,"人：名单里阳性",top[TARGET].sum(),"人")
# 打印两个指标：名单内阳性占比、这份名单抓到了全部阳性病人的比例
print("名单阳性占比 %.3f；抓住全部阳性的 %.3f"
     %(top[TARGET].mean(),top[TARGET].sum()/valid[TARGET].sum()))
# 取出需要交付的字段：就诊id、居民id、就诊时间、风险分数，只展示前5行预览
top[['visit_id','resident_id','visit_date','score']].head()

In [ ]:
# 5.6 保存模型、特征清单、指标
# 使用joblib保存最优的整条流水线Pipeline（包含预处理+随机森林模型）到model.pkl文件
# 保存最优的模型
joblib.dump(best_pipe,'model.pkl')
# 把特征名称列表FEATURES保存到feature_cols.json，utf-8编码，中文不乱码
json.dump(FEATURES,open('feature_cols.json','w',encoding='utf-8'),ensure_ascii=False,indent=2)
# 把逻辑回归、随机森林两个模型的评估指标results保存到metrics.json
# 作用：留存模型对比记录，方便写报告、版本追溯
json.dump(results,open('metrics.json','w',encoding='utf-8'),ensure_ascii=False,indent=2)
# 将筛选出来的15%高危复查名单，保存为csv文件，不写入pandas自带行索引
# 这个csv就是交付给业务/疾控的复查名单
top[['visit_id','resident_id','visit_date','score']].to_csv('top15.csv',index=False)
# 打印选出的最优模型标识，rf代表随机森林
print('best',best)

---

## 交付物

现在的状态：

| 文件 | 说明 |
|---|---|
| `train.parquet` / `valid.parquet` | 55,793 / 23,912 行，37 列，随机切分，阳性率都是 29.9% |
| `model.pkl` | 随机森林，整条 Pipeline，可以直接加载 |
| `feature_cols.json` | 25 列：17 个数值、8 个类别 |
| `metrics.json` | 逻辑回归 AUC 0.9406，随机森林 AUC 0.9468，准确率 96% |
| `top15.csv` | 3,587 人，名单里阳性占 96.8% |
| `predict.py` | 一条记录进，一个风险分出 |

### 5.8 扩展：使用AI，如何提示AI 模型

到目前位置，生成的文件。

【提示词】我有一个 SQLite 数据库 `incident.db`，里面有三张表：`residents`（居民档案）、`visits`（随访记录）、`lab_results`（化验结果）。请帮我写一个 Python 脚本：
1. 读取三张表，做数据清洗（缺失值、异常值、重复值、类别不统一）；
2. 构造标签：随访后 90 天内的化验中，空腹血糖 >= 7.0 或糖化血红蛋白 >= 6.5 记为 1，否则为 0；
3. 做特征工程（年龄、BMI 等），类别变量独热编码；
4. 划分训练集和验证集，保存成 `train.parquet` 和 `valid.parquet`；
5. 训练逻辑回归和随机森林，输出准确率、F1、AUC、混淆矩阵和分类报告；
6. 保存最好的模型，并写一个预测函数。